# 04 · Attention variants: MHA, MQA, GQA and MLA
Level L1 · Part 1 · professional · ~15 min · builds on 03

## Goal
By the end of this lesson you can name the four attention designs (MHA, MQA, GQA, MLA), say how each one changes the number of things the KV cache stores, and say what each design gives up in exchange.

## Why it matters
- **Without it:** you assume every model caches one key and one value per query head. For a 70B model that guess is 8 times too big, and you would buy the wrong GPUs or set the wrong concurrency limit.
- **With it:** one number from the model config, `num_key_value_heads`, tells you how big the cache is, and the design tells you what quality or compute the model paid to make it smaller.
- **When you don't need it:** if you only call a hosted model and pay per token, the provider picks the design. Read it on the model card so you know what your provider runs, and move on.

## The idea
Attention reads the keys and values of every earlier token. The cache keeps them so they are not recomputed (lesson 01). The number of **KV heads** sets how many key and value vectors are stored per token and per layer. Each design picks a different KV head count. Think of a support desk with 64 agents (query heads). MHA gives each agent a private binder of past tickets. MQA gives all of them one shared binder. GQA gives each team of 8 agents its own binder. MLA keeps one short summary card per ticket.

## Picture
```mermaid
flowchart LR
  subgraph MHA["MHA: 4 query heads, 4 KV heads"]
    direction TB
    a1["Q1"] --> ak1["K1 V1"]
    a2["Q2"] --> ak2["K2 V2"]
    a3["Q3"] --> ak3["K3 V3"]
    a4["Q4"] --> ak4["K4 V4"]
  end
  subgraph GQA["GQA: 4 query heads, 2 KV heads"]
    direction TB
    b1["Q1, Q2"] --> bk1["K1 V1"]
    b2["Q3, Q4"] --> bk2["K2 V2"]
  end
  subgraph MQA["MQA: 4 query heads, 1 KV head"]
    direction TB
    c1["Q1 to Q4"] --> ck1["K1 V1"]
  end
```

## Explained
### MHA: one KV head per query head
Multi-head attention (MHA) is the original design. Every query head has its own keys and values. A 70B model with 64 query heads stores 64 KV heads per layer. It has the best detail per head and the biggest cache.


### MQA: one KV head for all query heads
Multi-query attention (MQA) keeps many query heads but shares one key head and one value head across all of them. Noam Shazeer's 2019 paper showed that this reduces the memory bandwidth needed for incremental decoding. The cache is 1/64 of MHA for the 70B shape. The cost is quality. The paper reported "minor quality degradation". The 2023 GQA paper says MQA "can lead to quality degradation", so the cost is real.


### GQA: a few groups of query heads
Grouped-query attention (GQA) splits the query heads into groups, and each group shares one key head and one value head. Llama 3.1 70B has 64 query heads and 8 KV heads, so each KV head serves a group of 8 query heads. The cache is 8 times smaller than MHA. The GQA paper reports quality close to MHA at a speed close to MQA, and it shows that an existing multi-head checkpoint can be converted by uptraining (a short extra training run after the change). This is why most current open models use GQA.


### MLA: cache a small latent instead of heads
Multi-head latent attention (MLA), from DeepSeek-V2, compresses the keys and values into a small latent vector per token per layer; the cache stores that latent plus a small positional key (512 + 64 = 576 numbers per layer in DeepSeek-V3). A latent is a short compressed vector. The latent's length is `kv_lora_rank` (512 in DeepSeek-V3). The positional key is the small part that carries token position; its length is `qk_rope_head_dim` (64 in DeepSeek-V3). DeepSeek-V2 reports a 93.3% smaller KV cache than DeepSeek 67B. MLA is part of the model's architecture, so you choose a model that uses it, not a serving flag. The attention computation changes too; the details are outside this course.


### How to tell the design from config.json
1. A `kv_lora_rank` field means MLA.
2. `num_key_value_heads` = 1 means MQA.
3. `num_key_value_heads` = `num_attention_heads` means MHA.
4. Anything in between is GQA, with `num_attention_heads / num_key_value_heads` query heads per group.

## Worked example: one 70B request under four designs
The Brightside request from lesson 03 has 1,600 prompt tokens and a 200-token answer, 1,800 tokens in all. The config values below are copied from the public Llama 3.1 and DeepSeek-V3 `config.json` files (only the numbers used here). Nothing is downloaded.

In [1]:
CONFIGS = {
    "Llama 3.1 8B": {"num_hidden_layers": 32, "num_attention_heads": 32,
                     "num_key_value_heads": 8, "head_dim": 128},
    "Llama 3.1 70B": {"num_hidden_layers": 80, "num_attention_heads": 64,
                      "num_key_value_heads": 8, "hidden_size": 8192},  # head size = hidden_size / query heads when head_dim is absent
    "DeepSeek-V3": {"num_hidden_layers": 61, "num_attention_heads": 128,
                    "num_key_value_heads": 128,
                    "kv_lora_rank": 512, "qk_rope_head_dim": 64},
}

def variant(c):
    """Name the attention design from the config fields."""
    if "kv_lora_rank" in c:
        return "MLA"
    kv, q = c["num_key_value_heads"], c["num_attention_heads"]
    if kv == 1:
        return "MQA"
    return "MHA" if kv == q else f"GQA ({q // kv} query heads per KV head)"

def kv_bytes_per_token(c, kv_heads=None, bytes_per_value=2):
    """Standard cache: 2 (keys and values) x layers x KV heads x head size x bytes."""
    head_dim = c.get("head_dim") or c["hidden_size"] // c["num_attention_heads"]
    kv = kv_heads or c["num_key_value_heads"]
    return 2 * c["num_hidden_layers"] * kv * head_dim * bytes_per_value

def mla_bytes_per_token(c, bytes_per_value=2):
    """MLA cache: one latent plus one positional key per layer."""
    return (c["kv_lora_rank"] + c["qk_rope_head_dim"]) * c["num_hidden_layers"] * bytes_per_value

Next, every config gets one line: its design and the cache it needs per token.

In [2]:
for name, c in CONFIGS.items():
    per = mla_bytes_per_token(c) if variant(c) == "MLA" else kv_bytes_per_token(c)
    print(f"{name:14} {variant(c):<34} {per / 1024:>9,.1f} KiB per token")

Llama 3.1 8B   GQA (4 query heads per KV head)        128.0 KiB per token
Llama 3.1 70B  GQA (8 query heads per KV head)        320.0 KiB per token
DeepSeek-V3    MLA                                     68.6 KiB per token


**What you should see:** the 8B and 70B lines are GQA with 4 and 8 query heads per KV head. The DeepSeek-V3 line is MLA, at 68.6 KiB per token. Its `num_key_value_heads` is 128, but that number does not describe the cache. This is misconception 1 below.

Now the 70B model under each design, with the same FP16 cache (a number format that uses 2 bytes per number). The three rows change only the KV head count.

In [3]:
c70 = CONFIGS["Llama 3.1 70B"]
TOKENS = 1_800
for label, kv in [("MHA", c70["num_attention_heads"]), ("GQA", c70["num_key_value_heads"]), ("MQA", 1)]:
    per = kv_bytes_per_token(c70, kv_heads=kv)
    print(f"{label}: {kv:>2} KV heads -> {per / 1024:>6,.0f} KiB per token, "
          f"{per * TOKENS / 2**20:>7,.1f} MiB for the 1,800-token request")

MHA: 64 KV heads ->  2,560 KiB per token, 4,500.0 MiB for the 1,800-token request
GQA:  8 KV heads ->    320 KiB per token,   562.5 MiB for the 1,800-token request
MQA:  1 KV heads ->     40 KiB per token,    70.3 MiB for the 1,800-token request


**What you should see:** MHA needs 2,560 KiB per token and 4,500 MiB for the request. GQA needs 320 KiB and 562.5 MiB, the same number as lesson 03. MQA needs 40 KiB and 70.3 MiB. Each step is a change in one number, the KV head count.

To compare MLA with MHA on the same footing, use DeepSeek-V3 with its 61 layers. MLA models do not store per-head keys and values, so this MHA figure is a simplified what-if, using head size 128 for both.

In [4]:
v3 = CONFIGS["DeepSeek-V3"]
mla = mla_bytes_per_token(v3)
mha_est = kv_bytes_per_token({**v3, "head_dim": 128}, kv_heads=128)
print(f"DeepSeek-V3 MLA: {mla / 1024:.1f} KiB per token")
print(f"128-head MHA estimate: {mha_est / 1024:,.0f} KiB per token, about {mha_est / mla:.0f} times larger")

DeepSeek-V3 MLA: 68.6 KiB per token
128-head MHA estimate: 3,904 KiB per token, about 57 times larger


**What you should see:** about 68.6 KiB per token for MLA against 3,904 KiB (3.8 MiB) for the MHA what-if. The 57 times ratio is our rough 128-head estimate. The 93.3% figure is a measured DeepSeek-V2 vs 67B result, so the two numbers are not meant to match. The gap is far larger than the GQA gap, because MLA stores a latent that is much narrower than 128 heads of 128 numbers.

## At work
- **DeepSeek-V3** uses MLA with `kv_lora_rank` 512 and `qk_rope_head_dim` 64 (its `config.json`). DeepSeek-V2 reports a 93.3% smaller KV cache than DeepSeek 67B (arXiv 2405.04434 abstract). The smaller cache is what lets a long-context model keep many requests in memory. → lesson 05 turns the per-token size into a concurrency limit.
- **Open models** in the RESEARCH.md table use 2 to 8 KV heads: Qwen2.5 1.5B uses 2, Llama 3.1 8B and 70B use 8, Mistral-7B uses 8. GQA is the common choice, so check `num_key_value_heads` before you size anything. → lesson 05.

- **MQA quality:** Shazeer (2019) reports minor quality loss, and the GQA paper (2023) says MQA can degrade quality. Teams that need the smallest cache and accept the loss use MQA; most keep GQA. Go deeper: MLA internals and the absorbed-matrix math are research-level and outside this course.

Sources: Shazeer, "Fast Transformer Decoding: One Write-Head is All You Need" (arXiv 1911.02150) · Ainslie et al., "GQA" (arXiv 2305.13245) · DeepSeek-V2 (arXiv 2405.04434) · config.json of Llama 3.1 and DeepSeek-V3 on Hugging Face · RESEARCH.md model table (config.json files) for the Qwen2.5 1.5B and Mistral-7B KV head counts.

## Common misconceptions
**1. "The number of heads in the config is the cache."** For MLA models the field `num_key_value_heads` can say 128, but the cache holds a latent, not 128 heads. Read the variant first (`variant()` above), then pick the formula.

In [5]:
wrong = kv_bytes_per_token({**v3, "head_dim": 128}, kv_heads=128)  # treats the MLA model as MHA
right = mla_bytes_per_token(v3)
print(f"wrong: {wrong / 1024:,.0f} KiB per token, right: {right / 1024:.1f} KiB per token")

wrong: 3,904 KiB per token, right: 68.6 KiB per token


**2. "The MLA formula works on any config."** It needs the latent fields. A GQA config does not have them, so the lookup fails. Run `variant()` first.

In [6]:
print(mla_bytes_per_token(CONFIGS["Llama 3.1 70B"]))  # KeyError: 'kv_lora_rank' is only in MLA configs

KeyError: 'kv_lora_rank'

**3. "FP8 is the same as MQA."** An FP8 cache halves the bytes per number. It does not remove heads. For the 70B GQA model, FP8 takes 320 KiB to 160 KiB per token. MQA takes it to 40 KiB, and the two levers multiply. The design is fixed by the model, and the precision is a setting you can change.

## Check yourself
1. A config has 32 query heads, 4 KV heads, 40 layers and head size 128 in FP16. Which design is it, and how many KiB per token does it cache?
2. Model A is MQA and model B is MHA with 32 heads, same layers and head size. Which one fits more requests in the same cache memory, and what does A give up?
3. A config reports `num_key_value_heads` = 128 and has `kv_lora_rank` = 512. Is its cache 128 heads? What is it instead?
4. A team moves its 70B model to an FP8 cache and says "now we have MQA-level memory". Is that right?
5. Why would a team pick GQA with 8 KV heads over MQA for a 70B model?

**Answers**

1. GQA: 32 / 4 = 8 query heads per KV head. Bytes = 2 × 40 × 4 × 128 × 2 = 81,920 B = 80 KiB per token.
2. A (MQA) fits more: its cache is 1/32 of B's. A gives up some quality, because one shared key and value head limits the different attention patterns the model can keep.
3. No. It is MLA: the cache holds one latent (512 numbers) plus a positional key (64 numbers) per layer, about 68.6 KiB per token for DeepSeek-V3's 61 layers.
4. No. FP8 halves each number (320 KiB to 160 KiB for 70B GQA). MQA removes heads (320 KiB to 40 KiB). They are different levers, and they combine.
5. GQA keeps quality close to MHA at a speed close to MQA (GQA paper), with a cache 8 times smaller than MHA. MQA is smaller still but can lose quality.

## Recap
- The cache stores keys and values for every KV head, every layer and every token. The KV head count sets its size.
- **MHA** stores one KV head per query head (biggest cache). **MQA** stores one (smallest, quality cost). **GQA** stores one per group of query heads (the middle). **MLA** stores one small latent per layer (much smaller cache; the model must be built for it).
- Read the design from `config.json`: `kv_lora_rank` = MLA; `num_key_value_heads` = 1 = MQA; equal to the query heads = MHA; in between = GQA.
- The design is part of the model. FP8 is a separate setting that halves the bytes per number. The one thing people get wrong is reading the query head count, or an MLA model's `num_key_value_heads`, as the cache size.